In [1]:
import os
import json
from dotenv import load_dotenv

load_dotenv()
# Configure environment variables
service_endpoint = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
key = os.getenv("AZURE_SEARCH_ADMIN_KEY")
index_name = os.getenv("AZURE_SEARCH_INDEX")

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_GPT4_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT_NAME")
AZURE_OPENAI_EMBEDDINGS_ADA_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_EMBEDDINGS_ADA_DEPLOYMENT_NAME")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
azure_openai_embedding_dimensions = 1536

In [2]:
import pandas as pd
from tenacity import retry, wait_random_exponential, stop_after_attempt
from openai import AzureOpenAI
from langchain.text_splitter import CharacterTextSplitter,RecursiveCharacterTextSplitter
from langchain_openai import AzureOpenAIEmbeddings
from langchain.document_loaders import PyPDFLoader
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SimpleField,
    SearchFieldDataType,
    SearchableField,
    SearchField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SemanticSearch,
    SearchIndex,
    AzureOpenAIVectorizer,
    AzureOpenAIParameters
)


from azure.identity import DefaultAzureCredential, get_bearer_token_provider


In [3]:
# Configure OpenAI API
aoai_client = AzureOpenAI(
  azure_endpoint = AZURE_OPENAI_ENDPOINT, 
  api_key=AZURE_OPENAI_API_KEY,  
  api_version=AZURE_OPENAI_API_VERSION
)
credential = AzureKeyCredential(key)

In [4]:
# Generate Document Embeddings using OpenAI Ada Model
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
# Function to generate embeddings for title and content fields, also used for query embeddings
def calc_embeddings(text):
    # model = "deployment_name"
    embeddings = aoai_client.embeddings.create(input = [text], model=AZURE_OPENAI_EMBEDDINGS_ADA_DEPLOYMENT_NAME).data[0].embedding
    return embeddings

In [5]:
# splitting into 1000 char long chunks with 30 char overlap
# split ["\n\n", "\n", " ", ""]
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=30,
)

documentName = "fabric data engineering documentation"
fileName = "./data/fabric-data-engineering.pdf"
loader = PyPDFLoader(fileName)
pages = loader.load_and_split(text_splitter=splitter)
print("Number of chunks: ", len(pages))

Number of chunks:  449


In [6]:

import uuid
df = pd.DataFrame(columns=['id','document_name', 'content', 'embedding'])
for page in pages:
    df.loc[len(df.index)] = [str(uuid.uuid4()), documentName, page.page_content, ""]  
df.head()

,id,document_name,content,embedding
0,10dbd081-b223-4cb4-ac45-118bf4b4d695,fabric data engineering documentation,Tell us about your PDF experience.\nData Engin...,
1,0573fce5-449d-43b6-b6c1-ce3aa8cde889,fabric data engineering documentation,Visualize notebooks\nDelta Lake\nｐ CONCEPT\nLa...,
2,f52895ea-1ccb-48f3-a260-3918bcc4d3d9,fabric data engineering documentation,ｐ \nLakehouse scenario overview\nｇ TUTORIAL\nS...,
3,c3ce76ba-dcce-4366-aa65-3154fed2396f,fabric data engineering documentation,What is Data engineering in Microsoft\nFabric?...,
4,63332543-5bc0-4c68-8f94-3c8b8dfb8eed,fabric data engineering documentation,"analytics, as well as machine learning and oth...",


In [7]:
# calculate the embeddings using openAI ada 
df["embedding"] = df.content.apply(lambda x: calc_embeddings(x))
df.to_csv('./data/fabric_data_engineering_embeddings.csv', index=False)
print(df.head(2))

                                     id  \
0  10dbd081-b223-4cb4-ac45-118bf4b4d695   
1  0573fce5-449d-43b6-b6c1-ce3aa8cde889   

                           document_name  \
0  fabric data engineering documentation   
1  fabric data engineering documentation   

                                             content  \
0  Tell us about your PDF experience.\nData Engin...   
1  Visualize notebooks\nDelta Lake\nｐ CONCEPT\nLa...   

                                           embedding  
0  [-0.002991203684359789, 0.004324258770793676, ...  
1  [-0.006339251529425383, -0.016122134402394295,...  


In [8]:
# Output embeddings to json file
output_path = os.path.join('.', 'data', 'fabric_data_engineering_embeddings.json')

with open(output_path, 'w') as f:
    df.to_json(f, orient='records', default_handler=str)

In [9]:
# Create a search index
index_client = SearchIndexClient(endpoint=service_endpoint, credential=credential)
fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True, sortable=True, filterable=True, facetable=True),
    SearchableField(name="document_name", type=SearchFieldDataType.String, sortable=True, filterable=True, facetable=True),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SearchField(name="embedding", type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True, vector_search_dimensions=1536, vector_search_profile_name="myHnswProfile")
]

# Configure the vector search configuration  
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="myHnsw"
        )
    ],
    profiles=[
        VectorSearchProfile(
            name="myHnswProfile",
            algorithm_configuration_name="myHnsw",
            vectorizer="myVectorizer"
        )
    ],
    vectorizers=[
        AzureOpenAIVectorizer(
            name="myVectorizer",
            azure_open_ai_parameters=AzureOpenAIParameters(
                resource_uri=AZURE_OPENAI_ENDPOINT,
                deployment_id=AZURE_OPENAI_EMBEDDINGS_ADA_DEPLOYMENT_NAME,
                model_name=AZURE_OPENAI_EMBEDDINGS_ADA_DEPLOYMENT_NAME,
                api_key=AZURE_OPENAI_API_KEY
            )
        )
    ]
)

semantic_config = SemanticConfiguration(
    name="my-semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        content_fields=[SemanticField(field_name="content")]
    )
)
# Create the semantic settings with the configuration
semantic_search = SemanticSearch(configurations=[semantic_config])

# Create the search index with the semantic settings
index = SearchIndex(name=index_name, fields=fields,vector_search=vector_search, semantic_search=semantic_search)
result = index_client.create_or_update_index(index)
print(f' {result.name} created')

 books4 created


In [10]:
from azure.search.documents import SearchClient
import json

# Upload some documents to the index
output_path = os.path.join('.', 'data', 'fabric_data_engineering_embeddings.json')
with open(output_path, 'r') as file:  
    documents = json.load(file)  
search_client = SearchClient(endpoint=service_endpoint, index_name=index_name, credential=credential)
result = search_client.upload_documents(documents)
print(f"Uploaded {len(documents)} chunks") 

Uploaded 449 chunks
